In [0]:
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, DateType

seguros_schema = StructType([
    StructField("nome_contratante", StringType(), True),
    StructField("nr_documento_contratante", StringType(), True),
    StructField("cidade_contratante", StringType(), True),
    StructField("estado_contratante", StringType(), True),
    StructField("numero_apolice", IntegerType(), True),     
    StructField("data_contratacao", DateType(), True),        
    StructField("valor_pagamento", DoubleType(), True),
    StructField("valor_premio", DoubleType(), True),
    StructField("nome_beneficiario", StringType(), True),
    StructField("status_apolice", StringType(), True),
    StructField("Cobertura1", StringType(), True),
    StructField("Valor Cob 1", DoubleType(), True),
    StructField("Cobertura2", StringType(), True),
    StructField("Valor Cob 2", DoubleType(), True),
    StructField("Cobertura3", StringType(), True),
    StructField("Valor Cob 3", DoubleType(), True),
    StructField("capital_segurado", DoubleType(), True)
])

sinistros_schema = StructType([
    StructField("nome_segurado", StringType(), True),
    StructField("tipo_sinistro", StringType(), True),
    StructField("nome_beneficiario", StringType(), True),
    StructField("valor_sinistro", DoubleType(), True), 
    StructField("quem_forma_beneficiados", StringType(), True),
    StructField("status_seguro", StringType(), True),
    StructField("regiao_sinistro", StringType(), True)
])

nomes_schema = StructType([
    StructField("name", StringType(), True),
    StructField("classification", StringType(), True),
    StructField("frequency_female", IntegerType(), True),
    StructField("frequency_male", IntegerType(), True),
    StructField("frequency_total", IntegerType(), True),
    StructField("ratio", DoubleType(), True),
    StructField("names", StringType(), True)
])

caminhoSeguros = "/Workspace/Users/actc@cesar.school/Grupo7-Setor-de-Seguros/data/processed/bronze/seguros.csv"
caminhoSinistros = "/Workspace/Users/actc@cesar.school/Grupo7-Setor-de-Seguros/data/processed/bronze/sinistros.csv"
caminhoNomes = "/Workspace/Users/actc@cesar.school/Grupo7-Setor-de-Seguros/data/raw/grupos.csv" 

In [0]:
df_seguros = spark.read.format("csv").option("header", "true").option("sep", ",")\
    .option("dateFormat", "yyyy-MM-dd").schema(seguros_schema).load(caminhoSeguros)

df_sinistros = spark.read.format("csv").option("header", "true").option("sep", ",")\
    .schema(sinistros_schema).load(caminhoSinistros)


def load_and_prep_names_db(spark):
    df_names_raw = spark.read.format("csv").option("header", "true").option("sep", ",")\
                           .schema(nomes_schema).load(caminhoNomes)
    
    df_names = df_names_raw.withColumn("first_name", F.explode(F.split(F.col("names"), "\\|")))
    return df_names.select(F.col("first_name"), F.col("classification").alias("SEXO"))

df_nomes = load_and_prep_names_db(spark)
df_nomes = df_nomes.filter(F.col("first_name") != "")

In [0]:
def normalize_string(col_name):
    """Upsercase, Trim, and remove Accents for robust matching."""
    c = F.upper(F.trim(col_name))
    src = "ÁÉÍÓÚÀÈÌÒÙÂÊÎÔÛÃÕÑÇÄËÏÖÜ"
    dst = "AEIOUAEIOUAEIOUAONCAEIOU"
    return F.translate(c, src, dst)

In [0]:
def process_insurance_data(df_seg, df_sin, df_names):
    
    df_seg = df_seg.withColumn("key_contratante", normalize_string(F.col("nome_contratante"))) 
    df_sin = df_sin.withColumn("key_segurado", normalize_string(F.col("nome_segurado")))

    df_seg_dedup = df_seg.dropDuplicates(["nome_contratante", "nome_beneficiario"])
    
    join_cond = [
        df_seg_dedup["key_contratante"] == df_sin["key_segurado"]
    ]
    
    df_joined = df_seg_dedup.join(df_sin, join_cond, "left") \
        .drop(df_sin["nome_segurado"]) \
        .drop(df_sin["nome_beneficiario"]) \
        .drop("key_contratante", "key_segurado")

    regiao_map = {
        "Norte": ["AC", "AP", "AM", "PA", "RO", "RR", "TO"],
        "Nordeste": ["AL", "BA", "CE", "MA", "PB", "PE", "PI", "RN", "SE"],
        "Centro-Oeste": ["DF", "GO", "MT", "MS"],
        "Sudeste": ["ES", "MG", "RJ", "SP"],
        "Sul": ["PR", "RS", "SC"]
    }
    region_expr = F.when(F.col("estado_contratante").isNull(), "Desconhecido")
    for region, states in regiao_map.items():
        region_expr = region_expr.when(F.col("estado_contratante").isin(states), region)
    region_expr = region_expr.otherwise("Outro")
    
    df_joined = df_joined.withColumn("REGIAO", region_expr)

    first_name_raw = F.split(F.col("nome_contratante"), " ")[0]
    df_joined = df_joined.withColumn("primeiro_nome_join", normalize_string(first_name_raw))
    
    print("--- Debug ---")
    df_joined.select("nome_contratante", "primeiro_nome_join").show(5, truncate=False)

    df_joined = df_joined.join(df_names, df_joined.primeiro_nome_join == df_names.first_name, "left") \
                         .drop("primeiro_nome_join", "first_name")
    
    df_joined = df_joined.fillna({"SEXO": "Desconhecido"})


    df_joined = df_joined.withColumn("TRIMESTRE", F.quarter(F.col("data_contratacao")))


    if df_joined.count() > 0:
        q_premio = df_joined.approxQuantile("valor_premio", [0.25, 0.75], 0.01)
        q_capital = df_joined.approxQuantile("capital_segurado", [0.25, 0.75], 0.01)
        
        if len(q_premio) >= 2: q1_premio, q3_premio = q_premio[0], q_premio[1]
        else: q1_premio, q3_premio = 0.0, 0.0
            
        if len(q_capital) >= 2: q1_capital, q3_capital = q_capital[0], q_capital[1]
        else: q1_capital, q3_capital = 0.0, 0.0

        print(f"Premio: Q1={q1_premio}, Q3={q3_premio}")
        print(f"Capital: Q1={q1_capital}, Q3={q3_capital}")
    else:
        q1_premio, q3_premio = 0.0, 0.0
        q1_capital, q3_capital = 0.0, 0.0
        print("!!!")

    df_joined = df_joined.withColumn("ACIMA_DE_3_QUARTIL_PREMIO", F.col("valor_premio") > q3_premio) \
                         .withColumn("ABAIXO_DE_1_QUARTIL_PREMIO", F.col("valor_premio") < q1_premio) \
                         .withColumn("ACIMA_DE_3_QUARTIL_CAPITAL", F.col("capital_segurado") > q3_capital) \
                         .withColumn("ABAIXO_DE_1_QUARTIL_CAPITAL", F.col("capital_segurado") < q1_capital)

    w_segurado = Window.partitionBy("nome_contratante")
    df_joined = df_joined.withColumn("QTD_ACIDENTES_POR_NOME_SEGURADO", F.count("tipo_sinistro").over(w_segurado))
    df_joined = df_joined.withColumn("QTD_ACIDENTES_POR_NOME_CONTRATANTE", F.col("QTD_ACIDENTES_POR_NOME_SEGURADO"))

    df_joined = df_joined.withColumn("RAZÃO_PAGAMENTO_PREMIO", F.col("valor_pagamento") / F.col("valor_premio")) \
                         .withColumn("RAZÃO_PAGAMENTO_CAPITAL", F.col("valor_pagamento") / F.col("capital_segurado"))

    cols_to_drop = ["numero_apolice", "nr_documento_contratante", "cidade_contratante"]
    df_final = df_joined.drop(*cols_to_drop)

    return df_final

In [0]:
df_processed = process_insurance_data(df_seguros, df_sinistros, df_nomes)

print("\n--- Schema ---")
df_processed.printSchema()

In [0]:
display(df_processed)